In [1]:
from Experiment.Classes.PhyKanWrapper import PhyKAN as PK
from Experiment.Classes.MLP import CustomMLP

import torch
from torch import Generator

import torchvision.datasets as datasets
from torchvision import transforms

from torch.utils.data import DataLoader
from torchvision import transforms, datasets

from torch.nn import CrossEntropyLoss

from PhyKAN_Util_BN import PhyKAN as CurPhyKAN

import sys
import random

#from PhyKanClass import PhyKAN as PK

In [2]:
device='cpu'
torch.set_default_device(device)

In [9]:

shape = [784, 30, 10]
    
pk = CurPhyKAN(shape,3,1,1)

#mlp = CustomMLP([784, 30, 10], device='cpu')

In [10]:
torch.randn((784,100))


tensor([[-0.7004, -1.6502, -1.1387,  ...,  1.4184,  1.1043, -0.4571],
        [ 0.1079,  0.7761,  0.4483,  ...,  0.1783, -0.6420, -0.0984],
        [-2.3262, -2.4428, -1.3078,  ..., -1.8939,  1.1701,  1.1866],
        ...,
        [-1.3047, -1.8692, -0.8998,  ...,  1.0922, -0.4467,  0.4815],
        [ 0.4407, -0.4413,  0.4394,  ..., -0.0665, -1.2527,  0.2996],
        [ 0.7401, -0.4587,  1.0715,  ..., -1.7634,  1.4025, -0.8816]])

In [11]:
print(pk)

PhyKAN(
  (lossfn): MSELoss()
)


In [12]:
# Tweaks to PhyKan
# Returned all data representations between layers
# Removed lambda from the loss in the train calculation


In [ ]:
# Data load & transform

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.flatten(x))
    ])
    #transforms.Normalize((0.5,), (0.5,))
    #transforms.Lambda(lambda x: x / 255 ),

target_transform = transforms.Compose([
    transforms.Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
    ])

generator = Generator(device='cpu')

mnist_trainset = datasets.MNIST(root='./data', train=True, download=True) #, target_transform=target_transform)
mnist_trainset, mnist_valset = torch.utils.data.random_split(mnist_trainset, [50000, 10000])
batch_size = 100
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform, target_transform=target_transform) 
train_loader = DataLoader(train_set, shuffle=True, batch_size=batch_size, generator=generator)

#loss_fn = CrossEntropyLoss()

learning_rate = 0.01
running_loss = 0

for batch_idx, (batch_images, batch_labels) in enumerate(train_loader):
    #batch_images = batch_images.unsqueeze(0).reshape(batch_size,784)
    batch_images, batch_labels = batch_images.to('cpu'), batch_labels.to('cpu')

    #print(batch_images.shape)

    #outputs = pk.forward(batch_images)


    # print([out.shape for out in outputs]) [torch.Size([1000, 30]), torch.Size([1000, 10])]

    #print(predictions.shape)

    print(batch_images.shape)
    print(batch_labels[0])
    
    outputs, loss_np, loss_p, _= pk.train(batch_images, batch_labels, penalty=0.1, lamda=learning_rate)
    predictions = outputs[-1]
    
    running_loss += loss_np
     
    #print(predictions.shape, predictions.dtype )  #torch.Size([1000, 10])
    #print(predictions) # torch.Size([1000, 10])
    #print(batch_labels.shape, batch_labels.dtype) #torch.Size([1000, 10])
    #print(batch_labels)
    
    predicted_indices = torch.argmax(predictions, dim=1)
    print(predicted_indices[0])
    batch_indices = torch.argmax(batch_labels, dim=1)

    correct = (predicted_indices == batch_indices).sum().item()
    total = batch_labels.size(0)
    accuracy = correct / total

    print(loss_np)
    print(accuracy)


torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])
tensor(8)
1.0454380512237549
0.14
torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
tensor(7)
1.028536081314087
0.1
torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])
tensor(4)
0.9633805751800537
0.23
torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])
tensor(8)
0.9857616424560547
0.15
torch.Size([100, 784])
tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor(1)
0.9146388173103333
0.29
torch.Size([100, 784])
tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0.])
tensor(7)
0.879435122013092
0.31
torch.Size([100, 784])
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor(5)
0.8747610449790955
0.31
torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
tensor(7)
0.8755507469177246
0.4
torch.Size([100, 784])
tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
tensor(7)
0.834468424320221
0.49
torch.Size([100, 784])
tensor([0., 0., 1., 0., 0., 

KeyboardInterrupt: 